# **TAHAP 3** *Case Retrieval*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load Case Base

In [ ]:
df = pd.read_csv("/content/data/processed/cases.csv")

df.head()

,case_id,no_perkara,tanggal,jenis_perkara,terdakwa,pasal,ringkasan_fakta,argumen_hukum,amar_putusan,word_count,char_count,text_full
0,case_026,352.0,lahir : 45 tahun 8 agustus 1979; 4,pengancaman,fathurrahman; 2.,pasal 335 ayat,"menimbang, bahwa terdakwa diajukan ke persidan...",pasal 335 ayat,NaN,5457,37202,a h agung repub a h agung republik indonesia m...
1,case_015,450.0,lahir : 50 tahun 1 juli 1975 4,pengancaman,muhadi 2.,"pasal 170 ayat , pasal 44 kuhp, pasal 55 ayat","menimbang, bahwa para terdakwa diajukan ke per...","pasal 170 ayat , pasal 44 kuhp, pasal 55 ayat",NaN,23271,162746,a h agung repub a h agung republik indonesia m...
2,case_023,319.0,30 maret 2018; terdakwa ditahan di rumah tahan...,pengancaman,murnawadi,pasal 335 ayat,"menimbang, bahwa berdasarkan surat dakwaan pen...",pasal 335 ayat ini;,ini;,5543,38219,a h agung repub a h agung republik indonesia m...
3,case_009,319.0,30 maret 2018; terdakwa ditahan di rumah tahan...,pengancaman,murnawadi,pasal 335 ayat,"menimbang, bahwa berdasarkan surat dakwaan pen...",pasal 335 ayat ini;,ini;,5543,38219,a h agung repub a h agung republik indonesia m...
4,case_001,747.0,lahir : 46 tahun 31 desember 1979 4,pengancaman,abdullah 2.,"pasal 3 ayat , pasal 335 ayat , pasal 448 ayat...","menimbang, bahwa terdakwa diajukan ke persidan...","pasal 3 ayat , pasal 335 ayat , pasal 448 ayat...",ini ;,10273,71216,a h agung repub a h agung republik indonesia m...


# Siapkan Data Teks

In [ ]:
df["combined_text"]=(

    df["ringkasan_fakta"]
    .fillna("")
    .astype(str)

    + " " +

    df["argumen_hukum"]
    .fillna("")
    .astype(str)

    + " " +

    df["pasal"]
    .fillna("")
    .astype(str)
)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

print("TRAIN:", len(train_df))
print("TEST :", len(test_df))
print("RATIO:", round(len(train_df)/len(df), 2))

(27, 13)
(7, 13)
TRAIN: 27
TEST : 7
RATIO: 0.79


# TF-IDF Vector

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000
)

train_matrix = vectorizer.fit_transform(
    train_df["combined_text"]
)

print("TFIDF:", train_matrix.shape)

TFIDF: (27, 996)


# Function Retrieval

In [ ]:
def retrieve_case(query, top_k=5):

    query = str(query).lower()

    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vec,
        train_matrix
    ).flatten()

    top_idx = scores.argsort()[::-1][:top_k]

    results = train_df.iloc[top_idx].copy()

    results["similarity_score"] = scores[top_idx]

    return results[
        [
            "case_id",
            "no_perkara",
            "terdakwa",
            "pasal",
            "similarity_score"
        ]
    ]

# TEST QUERY

In [ ]:
query = """
pemerasan disertai pengancaman
terdakwa meminta uang dengan ancaman
"""

retrieve_case(query, top_k=5)

,case_id,no_perkara,terdakwa,pasal,similarity_score
16,case_027,395.0,raden satrianggi nurdiyono alias anggi;,"pasal 365 ayat , pasal 368 ayat , pasal 368 ku...",0.086809
28,case_017,588.0,hasanudin alias kencrung,pasal 368 ayat,0.071177
9,case_014,739.0,ahmad jalaludin alias jalal,"pasal 335 ayat , pasal 368 ayat , pasal 368kuh...",0.068909
7,case_010,631.0,NaN,pasal 222 kuhap terdakwa harus dibebani pula u...,0.056362
11,case_021,631.0,NaN,pasal 222 kuhap terdakwa harus dibebani pula u...,0.056362


In [ ]:
test_queries = [

{
"id":1,
"query":"pemerasan",
"expected":"pemerasan"
},

{
"id":2,
"query":"pengancaman",
"expected":"pengancaman"
},

{
"id":3,
"query":"pemerasan dan pengancaman",
"expected":"pemerasan_dan_pengancaman"
},

{
"id":4,
"query":"meminta uang dengan ancaman",
"expected":"pemerasan_dan_pengancaman"
},

{
"id":5,
"query":"mengancam korban",
"expected":"pengancaman"
}

]